In [21]:
import numpy as np
import pandas as pd
import time
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from src import util
from src.markov import simple_markov
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score, mean_absolute_error
plays_df = pd.read_csv('data/pbp/pbp_2005_2025_plays.csv')
drives_df = pd.read_csv('data/pbp/pbp_2005_2025_drives.csv')

In [ ]:
# ============================================================
# Covariate-dependent Markov: NESTED covariates, WHOLE structure
#
# Rows (10): baseline_A (counting), baseline_B (no cov),
#            +score, +time, +home, +spread, +total, +rain, +snow, +wind
# Play-level (7): top1/3/5/10 acc, true/predicted first-down rate, first-down AUC
# Drive-level (8): score acc/actual/pred, td acc/actual/pred, final-state acc, points MAE
# predicted_first_down_rate = mean of P(next in first-down states) over test plays
# ============================================================
# ---------------- CONFIG ----------------
SPLIT_MODE     = 'cut2020'
MIN_NEXT_COUNT = 300
FIRST_DOWN_STATES = list(range(1, 61))
ABS_IDS    = list(util.ABSORBING_STATES_MAP.keys())
ABS_POINTS = {241: 7, 242: 3, 243: 0, 244: 0, 245: 0, 246: 0}
SCORE_IDS  = [241, 242]
TD_ID      = 241
MAX_SIM_PLAYS = 40
N_SIM_DRIVES  = 3000
SIM_MODE    = 'sample'
FULL_DRIVES = True
N_SIM_REPEATS = 1
RNG = np.random.RandomState(42)

NESTED_ORDER = ['score_differential', 'game_seconds_remaining', 'is_home',
                'spread_line', 'total_line', 'rain', 'snow', 'wind']
NESTED_LABEL = {'score_differential': '+score', 'game_seconds_remaining': '+time',
                'is_home': '+home', 'spread_line': '+spread', 'total_line': '+total',
                'rain': '+rain', 'snow': '+snow', 'wind': '+wind'}
BOOL_COVARS = {'rain', 'snow'}

DRIVE_COL = {'score_differential': 'start_score_differential',
             'game_seconds_remaining': 'start_time_rem',
             'spread_line': 'spread_line', 'total_line': 'total_line',
             'rain': 'rain', 'snow': 'snow', 'wind': 'wind'}
DRIVE_DEFAULT = {'score_differential': 0.0, 'game_seconds_remaining': 1800.0,
                 'spread_line': 0.0, 'total_line': 44.0,
                 'rain': 0.0, 'snow': 0.0, 'wind': 0.0}

# ---------------- SPLIT ----------------
train_list, test_list, test_drives_list = util.get_train_test_sets(
    plays_df, drives_df, split_mode=SPLIT_MODE, train_frac=0.75)
print(f"split_mode={SPLIT_MODE}, folds={len(train_list)}", flush=True)

# ---------------- state id <-> (down, ytg, field) ----------------
_, ydstogo_labels = util.get_ydstogo_cuts_labels()
_, field_labels   = util.get_field_position_cuts_labels()
state_values = util.get_state_values(
    ["down", "binned_ydstogo_id", "binned_field_position_id"],
    [[1, 2, 3, 4], ydstogo_labels, field_labels])
_, id_to_state, _ = util.generate_state_ids(state_values, util.ABSORBING_STATES)
N_YTG, N_FIELD = len(ydstogo_labels), len(field_labels)

def state_onehot(state_ids):
    downs = np.array([id_to_state[s][0] for s in state_ids])
    ytgs  = np.array([id_to_state[s][1] for s in state_ids])
    flds  = np.array([id_to_state[s][2] for s in state_ids])
    cols  = [(downs == d).astype(float) for d in [1, 2, 3, 4]]
    cols += [(ytgs == y).astype(float) for y in range(1, N_YTG + 1)]
    cols += [(flds == f).astype(float) for f in range(1, N_FIELD + 1)]
    return np.column_stack(cols)

def covar_matrix(df, covars):
    if not covars:
        return np.zeros((len(df), 0))
    cols = []
    for c in covars:
        if c == 'is_home':
            cols.append((df['posteam_type'] == 'home').astype(float).to_numpy())
        elif c in BOOL_COVARS:
            cols.append(df[c].astype(bool).astype(float).to_numpy())
        else:
            cols.append(df[c].to_numpy(float))
    return np.column_stack(cols)

def covar_vec_drive(row, covars):
    if not covars:
        return np.zeros(0)
    vals = []
    for c in covars:
        if c == 'is_home':
            vals.append(1.0 if row.get('posteam_type') == 'home' else 0.0)
        else:
            v = row.get(DRIVE_COL[c], np.nan)
            if pd.isna(v):
                vals.append(DRIVE_DEFAULT[c])
            elif c in BOOL_COVARS:
                vals.append(1.0 if bool(v) else 0.0)
            else:
                vals.append(float(v))
    return np.array(vals, float)

# ---------------- metrics ----------------
def top_k_acc(proba, y_true, classes, k):
    topk = np.argsort(proba, axis=1)[:, -k:]
    ca = np.array(classes)
    return np.mean([y_true[i] in ca[topk[i]] for i in range(len(y_true))])

def play_metrics(proba_full, y_te, all_classes):
    fd_cols = [i for i, c in enumerate(all_classes) if c in FIRST_DOWN_STATES]
    fd_prob = proba_full[:, fd_cols].sum(axis=1)
    fd_true = np.isin(y_te, FIRST_DOWN_STATES).astype(int)
    return {
        'top_1_accuracy':  top_k_acc(proba_full, y_te, all_classes, 1),
        'top_3_accuracy':  top_k_acc(proba_full, y_te, all_classes, 3),
        'top_5_accuracy':  top_k_acc(proba_full, y_te, all_classes, 5),
        'top_10_accuracy': top_k_acc(proba_full, y_te, all_classes, 10),
        'true_first_down_rate':      fd_true.mean(),
        'predicted_first_down_rate': fd_prob.mean(),
        'first_down_roc_auc': roc_auc_score(fd_true, fd_prob)
                              if 0 < fd_true.sum() < len(fd_true) else np.nan,
    }

def drive_metrics(actual, preds):
    a_score = np.isin(actual, SCORE_IDS); p_score = np.isin(preds, SCORE_IDS)
    a_td = actual == TD_ID;               p_td = preds == TD_ID
    return {
        'drive_score_accuracy':       accuracy_score(a_score, p_score),
        'drive_score_actual_rate':    a_score.mean(),
        'drive_score_predicted_rate': p_score.mean(),
        'drive_td_accuracy':          accuracy_score(a_td, p_td),
        'drive_td_actual_rate':       a_td.mean(),
        'drive_td_predicted_rate':    p_td.mean(),
        'drive_final_state_accuracy': accuracy_score(actual, preds),
        'drive_points_mae': mean_absolute_error(
            [ABS_POINTS.get(a, 0) for a in actual], [ABS_POINTS.get(p, 0) for p in preds]),
    }

def average_dicts(ds):
    return {k: float(np.mean([d[k] for d in ds])) for k in ds[0]}

# ---------------- per-fold evaluation ----------------
def eval_fold(train_df, test_df, test_drives_df):
    base_req = ['cur_state_id', 'next_state_id', 'posteam_type',
                'score_differential', 'game_seconds_remaining',
                'spread_line', 'total_line', 'rain', 'snow', 'wind']
    tr = train_df.dropna(subset=base_req).copy()
    te = test_df.dropna(subset=base_req).copy()
    tr = tr[tr['cur_state_id'].isin(id_to_state.keys())]
    te = te[te['cur_state_id'].isin(id_to_state.keys())]
    vc = tr['next_state_id'].value_counts()
    keep = set(vc[vc >= MIN_NEXT_COUNT].index)
    tr = tr[tr['next_state_id'].isin(keep)]
    te = te[te['next_state_id'].isin(keep)]
    print(f"    train={len(tr):,} test={len(te):,} classes={len(keep)}", flush=True)

    dd = test_drives_df.dropna(subset=['start_state_id', 'drive_result_id']).copy()
    dd = dd[dd['start_state_id'].isin([s for s in id_to_state if s <= 240])]
    sim_drives = dd if FULL_DRIVES else dd.sample(n=min(N_SIM_DRIVES, len(dd)), random_state=1)
    s0_arr = sim_drives['start_state_id'].values.astype(int)
    actual = sim_drives['drive_result_id'].values.astype(int)
    cov_rows = sim_drives.to_dict('records')
    print(f"    drives to simulate: {len(s0_arr):,}", flush=True)

    te_states = te['cur_state_id'].values.astype(int)
    y_te = te['next_state_id'].values
    all_classes = sorted(keep)
    cls_idx = {c: i for i, c in enumerate(all_classes)}
    abs_set = set(ABS_IDS)
    Xs_te = state_onehot(te_states)
    Xs_tr = state_onehot(tr['cur_state_id'].values.astype(int))
    y_tr = tr['next_state_id'].values

    results = {}

    # ===== BASELINE A: counting =====
    t0 = time.time()
    print(f"    === baseline_A (counting) ===", flush=True)
    P = simple_markov.fit_markov_chain(tr)
    P_idx = {s: i for i, s in enumerate(P.index)}
    P_np = P.to_numpy(); P_cols = np.array(P.columns)
    proba_full = np.zeros((len(te_states), len(all_classes)))
    for sid in np.unique(te_states):
        if sid not in P_idx: continue
        rows = np.where(te_states == sid)[0]
        prow = P_np[P_idx[sid]]
        for j, c in enumerate(P_cols):
            if c in cls_idx: proba_full[rows, cls_idx[c]] = prow[j]
    proba_full = np.where(proba_full.sum(axis=1, keepdims=True) > 0,
                          proba_full, 1.0 / len(all_classes))
    proba_full /= proba_full.sum(axis=1, keepdims=True)
    res = play_metrics(proba_full, y_te, all_classes)
    dms = []
    for rep in range(N_SIM_REPEATS):
        preds = np.empty(len(s0_arr), dtype=int)
        for i in range(len(s0_arr)):
            state = s0_arr[i]
            for _ in range(MAX_SIM_PLAYS):
                if state in abs_set or state not in P_idx: break
                prow = P_np[P_idx[state]]
                if prow.sum() <= 0: break
                state = int(P_cols[np.argmax(prow)]) if SIM_MODE == 'greedy' \
                        else int(RNG.choice(P_cols, p=prow / prow.sum()))
            preds[i] = state if state in abs_set else 246
        dms.append(drive_metrics(actual, preds))
    res.update(average_dicts(dms))
    results['baseline_A_counting'] = res
    print(f"    done in {time.time()-t0:.1f}s (top1={res['top_1_accuracy']:.3f}, "
          f"ptsMAE={res['drive_points_mae']:.3f})", flush=True)

    # ===== multinomial WHOLE =====
    def run_whole(label, covars):
        t0 = time.time()
        print(f"    === {label} | covars={covars} ===", flush=True)
        X_tr = np.column_stack([Xs_tr, covar_matrix(tr, covars)])
        scaler = StandardScaler().fit(X_tr)
        clf = LogisticRegression(max_iter=1000, solver='saga', tol=1e-3)
        clf.fit(scaler.transform(X_tr), y_tr)
        classes = clf.classes_

        X_te = np.column_stack([Xs_te, covar_matrix(te, covars)])
        Pm = clf.predict_proba(scaler.transform(X_te))
        proba_full = np.zeros((len(te_states), len(all_classes)))
        for j, c in enumerate(classes):
            if c in cls_idx: proba_full[:, cls_idx[c]] = Pm[:, j]
        proba_full /= proba_full.sum(axis=1, keepdims=True).clip(min=1e-12)
        res = play_metrics(proba_full, y_te, all_classes)

        dms = []
        for rep in range(N_SIM_REPEATS):
            preds = np.empty(len(s0_arr), dtype=int)
            for i in range(len(s0_arr)):
                state = s0_arr[i]; cov_vec = covar_vec_drive(cov_rows[i], covars)
                for _ in range(MAX_SIM_PLAYS):
                    if state in abs_set: break
                    X = np.column_stack([state_onehot([state]), cov_vec[None, :]])
                    p = clf.predict_proba(scaler.transform(X))[0]
                    state = int(classes[np.argmax(p)]) if SIM_MODE == 'greedy' \
                            else int(RNG.choice(classes, p=p))
                preds[i] = state if state in abs_set else 246
                if (i + 1) % 5000 == 0:
                    print(f"          rep{rep+1} {i+1:,}/{len(s0_arr):,}", flush=True)
            dms.append(drive_metrics(actual, preds))
        res.update(average_dicts(dms))
        print(f"    done in {time.time()-t0:.1f}s (top1={res['top_1_accuracy']:.3f}, "
              f"ptsMAE={res['drive_points_mae']:.3f})", flush=True)
        return res

    results['baseline_B_nocov'] = run_whole('baseline_B', [])
    covars = []
    for c in NESTED_ORDER:
        covars = covars + [c]
        results[NESTED_LABEL[c]] = run_whole(NESTED_LABEL[c], list(covars))
    return results

# ---------------- run ----------------
all_folds = []
for k, (a, b, c) in enumerate(zip(train_list, test_list, test_drives_list)):
    print(f"  fold {k+1}/{len(train_list)}", flush=True)
    all_folds.append(eval_fold(a, b, c))

# ---------------- report: two tables ----------------
ROWS = ['baseline_A_counting', 'baseline_B_nocov'] + [NESTED_LABEL[c] for c in NESTED_ORDER]
PLAY_COLS = ['top_1_accuracy', 'top_3_accuracy', 'top_5_accuracy', 'top_10_accuracy',
             'true_first_down_rate', 'predicted_first_down_rate', 'first_down_roc_auc']
DRIVE_COLS = ['drive_score_accuracy', 'drive_score_actual_rate', 'drive_score_predicted_rate',
              'drive_td_accuracy', 'drive_td_actual_rate', 'drive_td_predicted_rate',
              'drive_final_state_accuracy', 'drive_points_mae']

summary = pd.DataFrame(
    {m: {mt: np.nanmean([f[m][mt] for f in all_folds]) for mt in PLAY_COLS + DRIVE_COLS}
     for m in ROWS}).T

pd.set_option('display.width', 220)
print("\n" + "=" * 100)
print(f"PLAY-LEVEL (10 x 7)   split={SPLIT_MODE}, sim={SIM_MODE}, repeats={N_SIM_REPEATS}")
print("=" * 100)
print(summary[PLAY_COLS].round(4).to_string())
print("\n" + "=" * 100)
print(f"DRIVE-LEVEL (10 x 8)")
print("=" * 100)
print(summary[DRIVE_COLS].round(4).to_string())

import os
os.makedirs('plots', exist_ok=True)
summary[PLAY_COLS].round(4).to_csv('plots/nested_play_level.csv')
summary[DRIVE_COLS].round(4).to_csv('plots/nested_drive_level.csv')
print("\nsaved: plots/nested_play_level.csv, plots/nested_drive_level.csv")

In [ ]:
# ============================================================
# Covariate-dependent Markov: NESTED covariate comparison, WHOLE structure only
#
# Baselines (no covariate):
#   A) counting     : fit_markov_chain, simulate from P rows
#   B) multinomial  : state one-hot only (= nested "core")
#
# Nested covariate sets (cumulative, fixed order), whole structure:
#   +score, +time, +home, +spread, +total
#
# Total = 2 baselines + 5 nested = 7 runs
# ============================================================
import numpy as np
import pandas as pd
import time
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score, mean_absolute_error
from src import util
from src.markov import simple_markov

# ---------------- CONFIG ----------------
SPLIT_MODE     = 'cut2020'
MIN_NEXT_COUNT = 300
FIRST_DOWN_STATES = list(range(1, 61))
ABS_IDS    = list(util.ABSORBING_STATES_MAP.keys())
ABS_POINTS = {241: 7, 242: 3, 243: 0, 244: 0, 245: 0, 246: 0}
MAX_SIM_PLAYS = 40
N_SIM_DRIVES  = 3000
SIM_MODE    = 'sample'
FULL_DRIVES = True
N_SIM_REPEATS = 1          # >1 averages drive-level metrics over repeated simulations
RNG = np.random.RandomState(42)

NESTED_ORDER = ['score_differential', 'game_seconds_remaining',
                'is_home', 'spread_line', 'total_line']
NESTED_LABEL = {'score_differential': '+score', 'game_seconds_remaining': '+time',
                'is_home': '+home', 'spread_line': '+spread', 'total_line': '+total'}

DRIVE_COL = {'score_differential': 'start_score_differential',
             'game_seconds_remaining': 'start_time_rem',
             'spread_line': 'spread_line', 'total_line': 'total_line'}
DRIVE_DEFAULT = {'score_differential': 0.0, 'game_seconds_remaining': 1800.0,
                 'spread_line': 0.0, 'total_line': 44.0}

# ---------------- SPLIT ----------------
train_list, test_list, test_drives_list = util.get_train_test_sets(
    plays_df, drives_df, split_mode=SPLIT_MODE, train_frac=0.75)
print(f"split_mode={SPLIT_MODE}, folds={len(train_list)}", flush=True)

# ---------------- state id <-> (down, ytg, field) ----------------
_, ydstogo_labels = util.get_ydstogo_cuts_labels()
_, field_labels   = util.get_field_position_cuts_labels()
state_values = util.get_state_values(
    ["down", "binned_ydstogo_id", "binned_field_position_id"],
    [[1, 2, 3, 4], ydstogo_labels, field_labels])
_, id_to_state, _ = util.generate_state_ids(state_values, util.ABSORBING_STATES)
N_YTG, N_FIELD = len(ydstogo_labels), len(field_labels)

def state_onehot(state_ids):
    downs = np.array([id_to_state[s][0] for s in state_ids])
    ytgs  = np.array([id_to_state[s][1] for s in state_ids])
    flds  = np.array([id_to_state[s][2] for s in state_ids])
    cols  = [(downs == d).astype(float) for d in [1, 2, 3, 4]]
    cols += [(ytgs == y).astype(float) for y in range(1, N_YTG + 1)]
    cols += [(flds == f).astype(float) for f in range(1, N_FIELD + 1)]
    return np.column_stack(cols)

def covar_matrix(df, covars):
    if not covars:
        return np.zeros((len(df), 0))
    cols = []
    for c in covars:
        if c == 'is_home':
            cols.append((df['posteam_type'] == 'home').astype(float).to_numpy())
        else:
            cols.append(df[c].to_numpy(float))
    return np.column_stack(cols)

def covar_vec_drive(row, covars):
    if not covars:
        return np.zeros(0)
    vals = []
    for c in covars:
        if c == 'is_home':
            vals.append(1.0 if row.get('posteam_type') == 'home' else 0.0)
        else:
            v = row.get(DRIVE_COL[c], np.nan)
            vals.append(DRIVE_DEFAULT[c] if pd.isna(v) else float(v))
    return np.array(vals, float)

def top_k_acc(proba, y_true, classes, k):
    topk = np.argsort(proba, axis=1)[:, -k:]
    ca = np.array(classes)
    return np.mean([y_true[i] in ca[topk[i]] for i in range(len(y_true))])

def play_metrics(proba_full, y_te, all_classes):
    ll  = log_loss(y_te, proba_full, labels=all_classes)
    t5  = top_k_acc(proba_full, y_te, all_classes, 5)
    t10 = top_k_acc(proba_full, y_te, all_classes, 10)
    fd_cols = [i for i, c in enumerate(all_classes) if c in FIRST_DOWN_STATES]
    fd_prob = proba_full[:, fd_cols].sum(axis=1)
    fd_true = np.isin(y_te, FIRST_DOWN_STATES).astype(int)
    fd_auc = roc_auc_score(fd_true, fd_prob) if 0 < fd_true.sum() < len(fd_true) else np.nan
    return {'state_log_loss': ll, 'top5_acc': t5, 'top10_acc': t10, 'first_down_roc_auc': fd_auc}

def drive_metrics(actual, preds):
    return {
        'drive_final_acc': accuracy_score(actual, preds),
        'drive_score_acc': accuracy_score(np.isin(actual,[241,242]), np.isin(preds,[241,242])),
        'drive_td_acc': accuracy_score(actual == 241, preds == 241),
        'drive_points_mae': mean_absolute_error(
            [ABS_POINTS.get(a,0) for a in actual], [ABS_POINTS.get(p,0) for p in preds]),
    }

def average_drive_metrics(list_of_dicts):
    keys = list_of_dicts[0].keys()
    return {k: float(np.mean([d[k] for d in list_of_dicts])) for k in keys}

# ---------------- per-fold evaluation ----------------
def eval_fold(train_df, test_df, test_drives_df):
    base_req = ['cur_state_id', 'next_state_id', 'posteam_type',
                'score_differential', 'game_seconds_remaining', 'spread_line', 'total_line']
    tr = train_df.dropna(subset=base_req).copy()
    te = test_df.dropna(subset=base_req).copy()
    tr = tr[tr['cur_state_id'].isin(id_to_state.keys())]
    te = te[te['cur_state_id'].isin(id_to_state.keys())]
    vc = tr['next_state_id'].value_counts()
    keep = set(vc[vc >= MIN_NEXT_COUNT].index)
    tr = tr[tr['next_state_id'].isin(keep)]
    te = te[te['next_state_id'].isin(keep)]
    print(f"    train={len(tr):,} test={len(te):,} classes={len(keep)}", flush=True)

    dd = test_drives_df.dropna(subset=['start_state_id', 'drive_result_id']).copy()
    dd = dd[dd['start_state_id'].isin([s for s in id_to_state if s <= 240])]
    sim_drives = dd if FULL_DRIVES else dd.sample(n=min(N_SIM_DRIVES, len(dd)), random_state=1)
    s0_arr = sim_drives['start_state_id'].values.astype(int)
    actual = sim_drives['drive_result_id'].values.astype(int)
    cov_rows = sim_drives.to_dict('records')

    te_states = te['cur_state_id'].values.astype(int)
    y_te = te['next_state_id'].values
    all_classes = sorted(keep)
    cls_idx = {c: i for i, c in enumerate(all_classes)}
    abs_set = set(ABS_IDS)
    Xs_te = state_onehot(te_states)

    results = {}

    # ======================= BASELINE A: counting =======================
    t0 = time.time()
    print(f"    === baseline A (counting) ===", flush=True)
    P = simple_markov.fit_markov_chain(tr)
    P_idx = {s: i for i, s in enumerate(P.index)}
    P_np = P.to_numpy(); P_cols = np.array(P.columns)
    proba_full = np.zeros((len(te_states), len(all_classes)))
    for sid in np.unique(te_states):
        if sid not in P_idx: continue
        rows = np.where(te_states == sid)[0]
        prow = P_np[P_idx[sid]]
        for j, c in enumerate(P_cols):
            if c in cls_idx: proba_full[rows, cls_idx[c]] = prow[j]
    proba_full = np.where(proba_full.sum(axis=1, keepdims=True) > 0,
                          proba_full, 1.0/len(all_classes))
    proba_full /= proba_full.sum(axis=1, keepdims=True)
    res = play_metrics(proba_full, y_te, all_classes)

    dm_list = []
    for rep in range(N_SIM_REPEATS):
        preds = np.empty(len(s0_arr), dtype=int)
        for i in range(len(s0_arr)):
            state = s0_arr[i]
            for _ in range(MAX_SIM_PLAYS):
                if state in abs_set or state not in P_idx: break
                prow = P_np[P_idx[state]]
                if prow.sum() <= 0: break
                state = int(P_cols[np.argmax(prow)]) if SIM_MODE == 'greedy' \
                        else int(RNG.choice(P_cols, p=prow / prow.sum()))
            preds[i] = state if state in abs_set else 246
        dm_list.append(drive_metrics(actual, preds))
    res.update(average_drive_metrics(dm_list))
    results['baseline_A_counting'] = res
    print(f"    done in {time.time()-t0:.1f}s (ll={res['state_log_loss']:.3f}, "
          f"ptsMAE={res['drive_points_mae']:.3f})", flush=True)

    # ======================= multinomial WHOLE runs =======================
    def run_whole(label, covars):
        t0 = time.time()
        print(f"    === {label} | whole | covars={covars} ===", flush=True)
        Xs_tr = state_onehot(tr['cur_state_id'].values.astype(int))
        Xc_tr = covar_matrix(tr, covars)
        X_tr = np.column_stack([Xs_tr, Xc_tr])
        scaler = StandardScaler().fit(X_tr)
        clf = LogisticRegression(max_iter=1000, solver='saga', tol=1e-3)
        clf.fit(scaler.transform(X_tr), tr['next_state_id'].values)
        classes = clf.classes_

        # play-level
        X_te = np.column_stack([Xs_te, covar_matrix(te, covars)])
        Pm = clf.predict_proba(scaler.transform(X_te))
        proba_full = np.zeros((len(te_states), len(all_classes)))
        for j, c in enumerate(classes):
            if c in cls_idx: proba_full[:, cls_idx[c]] = Pm[:, j]
        proba_full /= proba_full.sum(axis=1, keepdims=True).clip(min=1e-12)
        res = play_metrics(proba_full, y_te, all_classes)

        # drive-level (repeated simulations averaged)
        dm_list = []
        for rep in range(N_SIM_REPEATS):
            preds = np.empty(len(s0_arr), dtype=int)
            for i in range(len(s0_arr)):
                state = s0_arr[i]; cov_vec = covar_vec_drive(cov_rows[i], covars)
                for _ in range(MAX_SIM_PLAYS):
                    if state in abs_set: break
                    X = np.column_stack([state_onehot([state]), cov_vec[None, :]])
                    p = clf.predict_proba(scaler.transform(X))[0]
                    state = int(classes[np.argmax(p)]) if SIM_MODE == 'greedy' \
                            else int(RNG.choice(classes, p=p))
                preds[i] = state if state in abs_set else 246
                if (i + 1) % 5000 == 0:
                    print(f"          rep{rep+1} {i+1:,}/{len(s0_arr):,}", flush=True)
            dm_list.append(drive_metrics(actual, preds))
        res.update(average_drive_metrics(dm_list))
        print(f"    done in {time.time()-t0:.1f}s (ll={res['state_log_loss']:.3f}, "
              f"ptsMAE={res['drive_points_mae']:.3f})", flush=True)
        return res

    results['baseline_B_nocov'] = run_whole('baseline B', [])
    covars = []
    for c in NESTED_ORDER:
        covars = covars + [c]
        results[NESTED_LABEL[c]] = run_whole(NESTED_LABEL[c], list(covars))
    return results

# ---------------- run ----------------
all_folds = []
for k, (a, b, c) in enumerate(zip(train_list, test_list, test_drives_list)):
    print(f"  fold {k+1}/{len(train_list)}", flush=True)
    all_folds.append(eval_fold(a, b, c))

# ---------------- aggregate & report ----------------
order = ['baseline_A_counting', 'baseline_B_nocov'] + [NESTED_LABEL[c] for c in NESTED_ORDER]
mets = ['state_log_loss','top5_acc','top10_acc','first_down_roc_auc',
        'drive_final_acc','drive_score_acc','drive_td_acc','drive_points_mae']
summary = pd.DataFrame(
    {m: {mt: np.nanmean([f[m][mt] for f in all_folds]) for mt in mets} for m in order}).T

pd.set_option('display.width', 160)
print("\n" + "="*80)
print(f"NESTED COVARIATES — WHOLE structure  (split={SPLIT_MODE}, sim={SIM_MODE}, "
      f"repeats={N_SIM_REPEATS}, full_drives={FULL_DRIVES})")
print("="*80)
print("\n--- PLAY-LEVEL ---")
print(summary[['state_log_loss','top5_acc','top10_acc','first_down_roc_auc']].round(4).to_string())
print("\n--- DRIVE-LEVEL ---")
print(summary[['drive_final_acc','drive_score_acc','drive_td_acc','drive_points_mae']].round(4).to_string())

import os
os.makedirs('plots', exist_ok=True)
summary.round(4).to_csv('plots/covariate_nested_whole.csv')
print("\nsaved: plots/covariate_nested_whole.csv")

In [ ]:
# ============================================================
# Covariate-dependent Markov: NESTED covariate comparison
#
# Baselines (no covariate):
#   A) counting        : fit_markov_chain, P[i,j] = n_ij / n_i, simulate from P rows
#   B) multinomial     : state one-hot only, whole structure  (= nested "core")
#
# Nested covariate sets (cumulative, fixed order):
#   +score  : score_diff
#   +time   : + game_seconds_remaining
#   +home   : + is_home
#   +spread : + spread_line
#   +total  : + total_line
# each evaluated under 4 STRUCTURES:
#   whole / down / from_state_fallback / from_state_fitall
#
# Total = 2 baselines + 5 sets x 4 structures = 22 runs
# Primary drive metric: drive_points_mae ; secondary: drive_final_acc
# ============================================================
import numpy as np
import pandas as pd
import time
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score, mean_absolute_error
from src import util
from src.markov import simple_markov

# ---------------- CONFIG ----------------
SPLIT_MODE     = 'cut2020'
MIN_NEXT_COUNT = 300
RARE_MIN       = 100
FIRST_DOWN_STATES = list(range(1, 61))
ABS_IDS    = list(util.ABSORBING_STATES_MAP.keys())
ABS_POINTS = {241: 7, 242: 3, 243: 0, 244: 0, 245: 0, 246: 0}
MAX_SIM_PLAYS = 40
N_SIM_DRIVES  = 1000
SIM_MODE    = 'sample'
FULL_DRIVES = False
RNG = np.random.RandomState(42)

# nested order (fixed)
NESTED_ORDER = ['score_differential', 'game_seconds_remaining',
                'is_home', 'spread_line', 'total_line']
NESTED_LABEL = {'score_differential': '+score', 'game_seconds_remaining': '+time',
                'is_home': '+home', 'spread_line': '+spread', 'total_line': '+total'}
STRUCTURES = ['whole', 'down', 'from_state_fallback', 'from_state_fitall']

DRIVE_COL = {'score_differential': 'start_score_differential',
             'game_seconds_remaining': 'start_time_rem',
             'spread_line': 'spread_line', 'total_line': 'total_line'}
DRIVE_DEFAULT = {'score_differential': 0.0, 'game_seconds_remaining': 1800.0,
                 'spread_line': 0.0, 'total_line': 44.0}

# ---------------- SPLIT ----------------
train_list, test_list, test_drives_list = util.get_train_test_sets(
    plays_df, drives_df, split_mode=SPLIT_MODE, train_frac=0.75)
print(f"split_mode={SPLIT_MODE}, folds={len(train_list)}", flush=True)

# ---------------- state id <-> (down, ytg, field) ----------------
_, ydstogo_labels = util.get_ydstogo_cuts_labels()
_, field_labels   = util.get_field_position_cuts_labels()
state_values = util.get_state_values(
    ["down", "binned_ydstogo_id", "binned_field_position_id"],
    [[1, 2, 3, 4], ydstogo_labels, field_labels])
_, id_to_state, _ = util.generate_state_ids(state_values, util.ABSORBING_STATES)
N_YTG, N_FIELD = len(ydstogo_labels), len(field_labels)

def state_onehot(state_ids):
    downs = np.array([id_to_state[s][0] for s in state_ids])
    ytgs  = np.array([id_to_state[s][1] for s in state_ids])
    flds  = np.array([id_to_state[s][2] for s in state_ids])
    cols  = [(downs == d).astype(float) for d in [1, 2, 3, 4]]
    cols += [(ytgs == y).astype(float) for y in range(1, N_YTG + 1)]
    cols += [(flds == f).astype(float) for f in range(1, N_FIELD + 1)]
    return np.column_stack(cols)

def covar_matrix(df, covars):
    if not covars:
        return np.zeros((len(df), 0))
    cols = []
    for c in covars:
        if c == 'is_home':
            cols.append((df['posteam_type'] == 'home').astype(float).to_numpy())
        else:
            cols.append(df[c].to_numpy(float))
    return np.column_stack(cols)

def covar_vec_drive(row, covars):
    if not covars:
        return np.zeros(0)
    vals = []
    for c in covars:
        if c == 'is_home':
            vals.append(1.0 if row.get('posteam_type') == 'home' else 0.0)
        else:
            v = row.get(DRIVE_COL[c], np.nan)
            vals.append(DRIVE_DEFAULT[c] if pd.isna(v) else float(v))
    return np.array(vals, float)

def top_k_acc(proba, y_true, classes, k):
    topk = np.argsort(proba, axis=1)[:, -k:]
    ca = np.array(classes)
    return np.mean([y_true[i] in ca[topk[i]] for i in range(len(y_true))])

# ---------------- multinomial model container ----------------
class MarkovModel:
    def __init__(self, structure, covars):
        self.structure = structure
        self.covars = covars
        self.models = {}
        self.whole = None

    def _fit_one(self, Xs, Xc, y):
        X = np.column_stack([Xs, Xc])
        if len(np.unique(y)) < 2:
            return None
        scaler = StandardScaler().fit(X)
        clf = LogisticRegression(max_iter=1000, solver='saga', tol=1e-3)
        clf.fit(scaler.transform(X), y)
        return clf, scaler, clf.classes_

    def fit(self, tr):
        states = tr['cur_state_id'].values.astype(int)
        Xs = state_onehot(states); Xc = covar_matrix(tr, self.covars)
        y = tr['next_state_id'].values
        self.whole = self._fit_one(Xs, Xc, y)
        if self.structure == 'whole':
            return
        if self.structure == 'down':
            downs = np.array([id_to_state[s][0] for s in states])
            for d in [1, 2, 3, 4]:
                m = downs == d
                mdl = self._fit_one(Xs[m], Xc[m], y[m])
                if mdl: self.models[d] = mdl
            return
        counts = pd.Series(states).value_counts()
        fit_states = counts[counts >= RARE_MIN].index if self.structure == 'from_state_fallback' else counts.index
        n_fit = 0
        for i, sid in enumerate(fit_states):
            m = states == sid
            mdl = self._fit_one(Xs[m], Xc[m], y[m])
            if mdl:
                self.models[int(sid)] = mdl; n_fit += 1
            if (i + 1) % 50 == 0:
                print(f"          from_state {i+1}/{len(fit_states)}", flush=True)
        print(f"          {n_fit} state models fit, {len(fit_states)-n_fit} -> whole", flush=True)

    def _pick(self, state_id):
        if self.structure == 'whole':
            return self.whole
        if self.structure == 'down':
            return self.models.get(id_to_state[state_id][0], self.whole)
        return self.models.get(int(state_id), self.whole)

    def predict_one(self, state_id, cov_vec):
        clf, scaler, classes = self._pick(int(state_id))
        X = np.column_stack([state_onehot([state_id]), cov_vec[None, :]])
        return classes, clf.predict_proba(scaler.transform(X))[0]

# ---------------- shared evaluation helpers ----------------
def play_metrics_from_proba(proba_full, y_te, all_classes):
    ll  = log_loss(y_te, proba_full, labels=all_classes)
    t5  = top_k_acc(proba_full, y_te, all_classes, 5)
    t10 = top_k_acc(proba_full, y_te, all_classes, 10)
    fd_cols = [i for i, c in enumerate(all_classes) if c in FIRST_DOWN_STATES]
    fd_prob = proba_full[:, fd_cols].sum(axis=1)
    fd_true = np.isin(y_te, FIRST_DOWN_STATES).astype(int)
    fd_auc = roc_auc_score(fd_true, fd_prob) if 0 < fd_true.sum() < len(fd_true) else np.nan
    return {'state_log_loss': ll, 'top5_acc': t5, 'top10_acc': t10, 'first_down_roc_auc': fd_auc}

def drive_metrics(actual, preds):
    return {
        'drive_final_acc': accuracy_score(actual, preds),
        'drive_score_acc': accuracy_score(np.isin(actual,[241,242]), np.isin(preds,[241,242])),
        'drive_td_acc': accuracy_score(actual == 241, preds == 241),
        'drive_points_mae': mean_absolute_error(
            [ABS_POINTS.get(a,0) for a in actual], [ABS_POINTS.get(p,0) for p in preds]),
    }

# ---------------- per-fold evaluation ----------------
def eval_fold(train_df, test_df, test_drives_df):
    base_req = ['cur_state_id', 'next_state_id', 'posteam_type',
                'score_differential', 'game_seconds_remaining', 'spread_line', 'total_line']
    tr = train_df.dropna(subset=base_req).copy()
    te = test_df.dropna(subset=base_req).copy()
    tr = tr[tr['cur_state_id'].isin(id_to_state.keys())]
    te = te[te['cur_state_id'].isin(id_to_state.keys())]
    vc = tr['next_state_id'].value_counts()
    keep = set(vc[vc >= MIN_NEXT_COUNT].index)
    tr = tr[tr['next_state_id'].isin(keep)]
    te = te[te['next_state_id'].isin(keep)]
    print(f"    train={len(tr):,} test={len(te):,} classes={len(keep)}", flush=True)

    dd = test_drives_df.dropna(subset=['start_state_id', 'drive_result_id']).copy()
    dd = dd[dd['start_state_id'].isin([s for s in id_to_state if s <= 240])]
    sim_drives = dd if FULL_DRIVES else dd.sample(n=min(N_SIM_DRIVES, len(dd)), random_state=1)
    s0_arr = sim_drives['start_state_id'].values.astype(int)
    actual = sim_drives['drive_result_id'].values.astype(int)
    cov_rows = sim_drives.to_dict('records')

    te_states = te['cur_state_id'].values.astype(int)
    y_te = te['next_state_id'].values
    all_classes = sorted(keep)
    cls_idx = {c: i for i, c in enumerate(all_classes)}
    abs_set = set(ABS_IDS)
    Xs_te = state_onehot(te_states)

    results = {}

    # ======================= BASELINE A: counting =======================
    t0 = time.time()
    print(f"    === baseline A (counting) ===", flush=True)
    P = simple_markov.fit_markov_chain(tr)
    P_idx = {s: i for i, s in enumerate(P.index)}
    P_np = P.to_numpy()
    P_cols = np.array(P.columns)
    # play-level: P row for each test play
    proba_full = np.zeros((len(te_states), len(all_classes)))
    for sid in np.unique(te_states):
        if sid not in P_idx: continue
        rows = np.where(te_states == sid)[0]
        prow = P_np[P_idx[sid]]
        for j, c in enumerate(P_cols):
            if c in cls_idx:
                proba_full[rows, cls_idx[c]] = prow[j]
    proba_full = np.where(proba_full.sum(axis=1, keepdims=True) > 0,
                          proba_full, 1.0/len(all_classes))
    proba_full /= proba_full.sum(axis=1, keepdims=True)
    res = play_metrics_from_proba(proba_full, y_te, all_classes)
    # drive-level: sample from P rows
    preds = np.empty(len(s0_arr), dtype=int)
    for i in range(len(s0_arr)):
        state = s0_arr[i]
        for _ in range(MAX_SIM_PLAYS):
            if state in abs_set or state not in P_idx:
                break
            prow = P_np[P_idx[state]]
            if prow.sum() <= 0: break
            state = int(P_cols[np.argmax(prow)]) if SIM_MODE == 'greedy' \
                    else int(RNG.choice(P_cols, p=prow / prow.sum()))
        preds[i] = state if state in abs_set else 246
        if (i + 1) % 5000 == 0:
            print(f"          {i+1:,}/{len(s0_arr):,}", flush=True)
    res.update(drive_metrics(actual, preds))
    results[('baseline_A_counting', '-')] = res
    print(f"    done in {time.time()-t0:.1f}s (ll={res['state_log_loss']:.3f})", flush=True)

    # ======================= multinomial runs =======================
    def run_multinomial(label, structure, covars):
        t0 = time.time()
        print(f"    === {label} | {structure} | covars={covars} ===", flush=True)
        model = MarkovModel(structure, covars); model.fit(tr)
        te_cov = covar_matrix(te, covars)
        proba_full = np.zeros((len(te_states), len(all_classes)))
        for sid in np.unique(te_states):
            rows = np.where(te_states == sid)[0]
            clf, scaler, classes = model._pick(int(sid))
            X = np.column_stack([Xs_te[rows], te_cov[rows]])
            Pm = clf.predict_proba(scaler.transform(X))
            for j, c in enumerate(classes):
                if c in cls_idx: proba_full[rows, cls_idx[c]] = Pm[:, j]
        proba_full /= proba_full.sum(axis=1, keepdims=True).clip(min=1e-12)
        res = play_metrics_from_proba(proba_full, y_te, all_classes)
        preds = np.empty(len(s0_arr), dtype=int)
        for i in range(len(s0_arr)):
            state = s0_arr[i]; cov_vec = covar_vec_drive(cov_rows[i], covars)
            for _ in range(MAX_SIM_PLAYS):
                if state in abs_set: break
                classes, p = model.predict_one(state, cov_vec)
                state = int(classes[np.argmax(p)]) if SIM_MODE == 'greedy' \
                        else int(RNG.choice(classes, p=p))
            preds[i] = state if state in abs_set else 246
            if (i + 1) % 5000 == 0:
                print(f"          {i+1:,}/{len(s0_arr):,}", flush=True)
        res.update(drive_metrics(actual, preds))
        print(f"    done in {time.time()-t0:.1f}s (ll={res['state_log_loss']:.3f}, "
              f"ptsMAE={res['drive_points_mae']:.3f})", flush=True)
        return res

    # BASELINE B: multinomial, no covariate, whole
    results[('baseline_B_nocov', 'whole')] = run_multinomial('baseline B', 'whole', [])

    # NESTED: cumulative covariates x 4 structures
    covars = []
    for c in NESTED_ORDER:
        covars = covars + [c]
        label = NESTED_LABEL[c]
        for struct in STRUCTURES:
            results[(label, struct)] = run_multinomial(label, struct, list(covars))

    return results

# ---------------- run ----------------
all_folds = []
for k, (a, b, c) in enumerate(zip(train_list, test_list, test_drives_list)):
    print(f"  fold {k+1}/{len(train_list)}", flush=True)
    all_folds.append(eval_fold(a, b, c))

# ---------------- aggregate & report ----------------
keys = list(all_folds[0].keys())
mets = ['state_log_loss','top5_acc','top10_acc','first_down_roc_auc',
        'drive_final_acc','drive_score_acc','drive_td_acc','drive_points_mae']
agg = pd.DataFrame(
    {key: {m: np.nanmean([f[key][m] for f in all_folds]) for m in mets} for key in keys}).T
agg.index = pd.MultiIndex.from_tuples(agg.index, names=['model', 'structure'])

def pivot_metric(metric):
    rows = ['baseline_A_counting', 'baseline_B_nocov'] + [NESTED_LABEL[c] for c in NESTED_ORDER]
    tbl = pd.DataFrame(index=rows, columns=STRUCTURES, dtype=float)
    for (mdl, st), r in agg.iterrows():
        if mdl.startswith('baseline'):
            tbl.loc[mdl, 'whole'] = r[metric]
        else:
            tbl.loc[mdl, st] = r[metric]
    return tbl

pd.set_option('display.width', 160)
print("\n" + "="*80)
print(f"NESTED COVARIATE COMPARISON  (split={SPLIT_MODE}, sim={SIM_MODE}, rare<{RARE_MIN})")
print("rows: baselines + cumulative covariates ; cols: structure")
print("="*80)
for metric, note in [('drive_points_mae', 'PRIMARY, lower better'),
                     ('drive_final_acc', 'SECONDARY, higher better'),
                     ('state_log_loss', 'play-level, lower better'),
                     ('first_down_roc_auc', 'play-level, higher better')]:
    print(f"\n--- {metric}  ({note}) ---")
    print(pivot_metric(metric).round(4).to_string())

import os
os.makedirs('plots', exist_ok=True)
agg.round(4).to_csv('plots/covariate_nested_results.csv')
print("\nsaved: plots/covariate_nested_results.csv")

Getting train/test sets using split mode "cut2020" with training fraction of games = 0.75
split_mode=cut2020, folds=1
  fold 1/1
    train=601,105 test=196,066 classes=200
    === baseline A (counting) ===
Fitting Markov chain
    done in 4.7s (ll=2.322)
    === baseline B | whole | covars=[] ===
